# Faster Whisper Trial

In [18]:
from faster_whisper import WhisperModel
import pyaudio
import numpy as np

## Specify Model Parameters

In [19]:
model_size = "large-v3"

# Audio recording to fit the model
RATE = 16000
CHUNK = 1024
device = "cuda"
compute_type = "int8_float16" 

# List devices

In [12]:
p = pyaudio.PyAudio()
info = p.get_host_api_info_by_index(0)
numdevices = info.get('deviceCount')
for i in range(0, numdevices):
        if (p.get_device_info_by_host_api_device_index(0, i).get('maxInputChannels')) > 0:
            print(i, ". ", p.get_device_info_by_host_api_device_index(0, i).get('name'))

0 .  Microsoft Sound Mapper - Input
1 .  Microphone (Razer Seiren X)


# Load Mic

In [ ]:
p = pyaudio.PyAudio()
INPUT_DEVICE_INDEX = None # none means default device

stream = p.open(format=pyaudio.paInt16,
                channels=1,
                rate=RATE,
                input=True,
                input_device_index=INPUT_DEVICE_INDEX,
                frames_per_buffer=CHUNK)



Listening from microphone... Press Ctrl+C to stop.


In [21]:
model = WhisperModel(model_size, device=device, compute_type=compute_type)

In [25]:
try:
    print("Listening from microphone... Press Ctrl+C to stop.")
    while True:
        audio_data = b""
        for _ in range(0, int(RATE / CHUNK * 1)):  # Adjust the duration as needed
            audio_data += stream.read(CHUNK, exception_on_overflow=True)
        audio_np = np.frombuffer(audio_data, dtype=np.int16).astype(np.float32) / 32768.0
        segments, info = model.transcribe(audio_np, beam_size=5, language="en", task="transcribe")
        for segment in segments:
            print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))
except KeyboardInterrupt:
    print("Stopped listening.")

Listening from microphone... Press Ctrl+C to stop.
[0.00s -> 0.94s]  Hello.
[0.00s -> 1.00s]  My old friend
[0.00s -> 29.98s]  Thank you for watching!
[0.00s -> 29.98s]  Thank you.
[0.00s -> 0.96s]  I've come to-
[0.00s -> 0.98s]  To talk with you.
[0.00s -> 0.72s]  I love you again.
[0.00s -> 29.98s]  Thank you.
[0.00s -> 29.98s]  Thank you.
[0.00s -> 0.96s]  and no
[0.00s -> 0.72s]  hopefully
[0.00s -> 0.90s]  Creepy.
[0.00s -> 29.98s]  I love you.
[0.00s -> 29.98s]  Thank you.
[0.00s -> 2.00s]  Left it
[0.00s -> 0.84s]  See you tomorrow.
[0.00s -> 0.44s]  Flower.
[0.00s -> 0.88s]  Sleepy.
[0.00s -> 29.98s]  Thank you for watching.
Stopped listening.


In [17]:

stream.stop_stream()
stream.close()
p.terminate()